In [2]:
from langgraph.graph import StateGraph,START,END
from langchain_openai import ChatOpenAI
from typing import TypedDict
from dotenv import load_dotenv

In [3]:
load_dotenv()
model = ChatOpenAI()

In [15]:
class BlogState(TypedDict):
    title:str
    outline:str
    content:str
    evaluation:int

In [16]:
def create_outline(state:BlogState)->BlogState:
    # fetch title
    title = state['title']


    #call llm gen outline
    prompt = f"Generate a detailed outline for a blog on the topic - {title}"
    outline = model.invoke(prompt).content
    # update state
    state['outline'] = outline

    return state

In [17]:
def create_blog(state:BlogState)->BlogState:
    title = state['title']
    outline = state['outline']

    # llm call
    prompt = f"write a detailed blog on the title - {title} using the following outline : \n {outline}"
    content = model.invoke(prompt).content

    state['content'] = content

    return state

In [18]:
def blog_evaluate(state:BlogState)->BlogState:
    content = state['content']
    outline = state['outline']

    prompt = f"based on this outline : {outline} , rate my blog {content} and generate a integer score."
    evaluation = model.invoke(prompt).content

    state['evaluation'] = evaluation
    return state

In [19]:
graph  = StateGraph(BlogState)

# nodes
graph.add_node('create_outline',create_outline)
graph.add_node('create_blog',create_blog)
graph.add_node('blog_evaluate',blog_evaluate)

# edges
graph.add_edge(START,'create_outline')
graph.add_edge('create_outline','create_blog')
graph.add_edge('create_blog','blog_evaluate')
graph.add_edge('blog_evaluate',END)

workflow  =graph.compile()

In [21]:
initial_state = {'title':"Rise of AI in bangladesh"}

final_output = workflow.invoke(initial_state)

print(final_output['evaluation'])

I would rate this blog post an 8 out of 10. The outline is well-structured and covers all the key aspects of AI adoption in Bangladesh, including history, applications, challenges, opportunities, and future implications. The content is informative and insightful, providing a comprehensive overview of the topic. The call for continued research and investment in AI technology is also a strong conclusion. Overall, the blog post effectively addresses the rise of AI in Bangladesh and its potential impact on society and the economy. Great job!
